<div align="center">

  <h1><img src="https://raw.githubusercontent.com/auxeno/ion/main/assets/logo.png" alt="Ion" width="72"><br>GNN on BBBP</h1>

  [![Open in nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/Auxeno/ion/blob/main/examples/gnn_bbbp.ipynb)
  [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auxeno/ion/blob/main/examples/gnn_bbbp.ipynb)

</div>

---

Molecular property prediction with graph neural networks using [Ion](https://github.com/auxeno/ion).

Molecules are naturally graphs: atoms are nodes, bonds are edges. A graph neural network (GNN) learns by passing messages along these edges, letting each atom gather information from its neighbors. We use [GATv2Conv](https://arxiv.org/abs/2105.14491), an attention-based GNN layer that learns *which* neighbors and bonds matter most for a given prediction.

The task: predict whether a molecule can cross the blood-brain barrier (BBB), a selective membrane that protects the brain. The [BBBP](https://moleculenet.org/datasets-1) dataset has ~2,000 labeled molecules from MoleculeNet.

In [1]:
# !pip install ion-nn optax plotly tqdm rdkit

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from tqdm import tqdm

import ion
from ion import gnn, nn

RDLogger.DisableLog("rdApp.*")
pio.renderers.default = "notebook_connected"

## Load BBBP

BBBP (Blood-Brain Barrier Penetration) is a binary classification dataset. Each molecule is stored as a [SMILES](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) string, a compact text encoding of molecular structure (e.g. `C(=O)O` for formic acid). The label is 1 if the molecule crosses the blood-brain barrier, 0 if not.

In [3]:
import csv
import urllib.request
from pathlib import Path

cache_dir = Path.home() / ".cache" / "ion" / "bbbp"
cache_dir.mkdir(parents=True, exist_ok=True)
path = cache_dir / "BBBP.csv"

if not path.exists():
    print("Downloading BBBP...")
    urllib.request.urlretrieve(
        "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv",
        path,
    )

with open(path) as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Molecules: {len(rows)}")
for row in rows[:3]:
    print(f"  {row['smiles'][:50]:<50s}  label={row['p_np']}")

Molecules: 2050
  [Cl].CC(C)NCC(O)COc1cccc2ccccc12                    label=1
  C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl            label=1
  c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO3)=O  label=1


## SMILES to graph

We use [RDKit](https://www.rdkit.org/) to parse each SMILES string into a graph. Each atom becomes a node, and each bond becomes a pair of directed edges (one in each direction) since message passing happens both ways.

- Node features (9 dims): one-hot encoding of atomic number. Covers the 8 most common elements in drug molecules (C, N, O, F, P, S, Cl, Br) plus a catch-all for anything else.
- Edge features (4 dims): one-hot encoding of bond type (single, double, triple, aromatic).

In [4]:
# C, N, O, F, P, S, Cl, Br
ATOM_TYPES = [6, 7, 8, 9, 15, 16, 17, 35]

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]


def mol_to_graph(smiles):
    """Convert a SMILES string to (node_features, senders, receivers, edge_features) or None."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Node features: one-hot atomic number
    node_feats = []
    for atom in mol.GetAtoms():
        num = atom.GetAtomicNum()
        enc = [0] * (len(ATOM_TYPES) + 1)
        if num in ATOM_TYPES:
            enc[ATOM_TYPES.index(num)] = 1
        else:
            enc[-1] = 1
        node_feats.append(enc)
    x = np.array(node_feats, dtype=np.float32)

    if mol.GetNumBonds() == 0:
        empty = np.zeros(0, dtype=np.int32)
        return x, empty, empty, np.zeros((0, len(BOND_TYPES)), dtype=np.float32)

    # Edge features: one-hot bond type, both directions per bond
    senders, receivers, edge_feats = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = [1 if bond.GetBondType() == t else 0 for t in BOND_TYPES]
        senders.extend([i, j])
        receivers.extend([j, i])
        edge_feats.extend([feat, feat])

    return (
        x,
        np.array(senders, dtype=np.int32),
        np.array(receivers, dtype=np.int32),
        np.array(edge_feats, dtype=np.float32),
    )


result = mol_to_graph(rows[0]["smiles"])
x_test, s_test, r_test, e_test = result
print(f"SMILES:        {rows[0]['smiles']}")
print(f"Node features: {x_test.shape}")
print(f"Edges:         {s_test.shape[0]}")
print(f"Edge features: {e_test.shape}")

SMILES:        [Cl].CC(C)NCC(O)COc1cccc2ccccc12
Node features: (20, 9)
Edges:         40
Edge features: (40, 4)


## Build the full graph

All molecules are combined into one big disconnected graph. Each molecule's node indices are offset so they don't overlap. A `graph_index` array records which molecule each node belongs to. We use this later to pool node-level outputs into a single prediction per molecule.

We also add self-loops (each node connects to itself) so atoms retain their own features during message passing.

In [5]:
all_node_feats = []
all_senders = []
all_receivers = []
all_edge_feats = []
all_graph_index = []
mol_labels = []
mol_smiles = []
mol_names = []
node_offset = 0

for row in rows:
    result = mol_to_graph(row["smiles"])
    if result is None:
        continue

    x_mol, s_mol, r_mol, e_mol = result
    n = x_mol.shape[0]

    all_node_feats.append(x_mol)
    all_senders.append(s_mol + node_offset)
    all_receivers.append(r_mol + node_offset)
    all_edge_feats.append(e_mol)
    all_graph_index.append(np.full(n, len(mol_labels), dtype=np.int32))
    mol_labels.append(int(row["p_np"]))
    mol_smiles.append(row["smiles"])
    mol_names.append(row["name"])
    node_offset += n

x = jnp.array(np.concatenate(all_node_feats))
senders = jnp.array(np.concatenate(all_senders))
receivers = jnp.array(np.concatenate(all_receivers))
x_edge = jnp.array(np.concatenate(all_edge_feats))
graph_index = jnp.array(np.concatenate(all_graph_index))
labels = jnp.array(mol_labels, dtype=jnp.float32)

num_nodes = x.shape[0]
num_mols = len(mol_labels)

# Add self-loops so nodes aggregate their own features
senders, receivers = gnn.add_self_loops(senders, receivers, num_nodes)
x_edge = jnp.concatenate([x_edge, jnp.zeros((num_nodes, x_edge.shape[1]))])

print(f"Molecules:  {num_mols}")
print(f"Nodes:      {num_nodes:,}")
print(f"Edges:      {senders.shape[0]:,} (with self-loops)")
print(f"Node dim:   {x.shape[1]}")
print(f"Edge dim:   {x_edge.shape[1]}")
print(f"Positive:   {labels.mean():.1%}")

Molecules:  2039
Nodes:      49,068
Edges:      154,910 (with self-loops)
Node dim:   9
Edge dim:   4
Positive:   76.5%


## Visualize sample molecules

Molecules drawn as 2D graphs using RDKit for layout. Atom colors follow the [CPK convention](https://en.wikipedia.org/wiki/CPK_coloring) used in chemistry (carbon = dark gray, nitrogen = blue, oxygen = red, etc.). Double bonds are drawn as parallel lines, triple as three lines, and aromatic bonds as dotted lines.

In [6]:
ELEMENT_COLORS = {
    6: "#404040",   # Carbon - dark gray
    7: "#636EFA",   # Nitrogen - blue
    8: "#EF553B",   # Oxygen - red
    9: "#00CC96",   # Fluorine - green
    15: "#FFA15A",  # Phosphorus - orange
    16: "#FECB52",  # Sulfur - yellow
    17: "#19D3F3",  # Chlorine - cyan
    35: "#FF6692",  # Bromine - pink
}
OTHER_COLOR = "#AB63FA"
BOND_COLOR = "#888888"

ELEMENT_NAMES = ["C", "N", "O", "F", "P", "S", "Cl", "Br", "Other"]
ELEMENT_PALETTE = [ELEMENT_COLORS[n] for n in ATOM_TYPES] + [OTHER_COLOR]


def mol_coords(mol):
    """Compute 2D layout coordinates for a molecule."""
    AllChem.Compute2DCoords(mol)
    conf = mol.GetConformer()
    return np.array(
        [[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y]
         for i in range(mol.GetNumAtoms())]
    )


def draw_molecule(fig, mol, row, col, bond_weights=None, atom_weights=None, show_bond_types=True):
    """Draw a molecule on a Plotly subplot."""
    coords = mol_coords(mol)

    for idx, bond in enumerate(mol.GetBonds()):
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondType()

        if bond_weights is None:
            width, opacity = 1.5, 1.0
        else:
            w = bond_weights[idx]
            width = 1 + w * 6
            opacity = 0.3 + 0.7 * w

        # Perpendicular offset for drawing parallel lines (double/triple bonds)
        x0, y0 = coords[i]
        x1, y1 = coords[j]
        dx, dy = x1 - x0, y1 - y0
        length = np.hypot(dx, dy)
        if length > 0:
            px, py = -dy / length * 0.06, dx / length * 0.06
        else:
            px, py = 0, 0

        if not show_bond_types:
            offsets, dash = [0], "solid"
        elif bt == Chem.rdchem.BondType.DOUBLE:
            offsets, dash = [-1, 1], "solid"
        elif bt == Chem.rdchem.BondType.TRIPLE:
            offsets, dash = [-1, 0, 1], "solid"
        elif bt == Chem.rdchem.BondType.AROMATIC:
            offsets, dash = [0], "dot"
        else:
            offsets, dash = [0], "solid"

        for s in offsets:
            ox, oy = s * px, s * py
            fig.add_trace(go.Scatter(
                x=[x0 + ox, x1 + ox], y=[y0 + oy, y1 + oy],
                mode="lines",
                line=dict(color=BOND_COLOR, width=width, dash=dash),
                opacity=opacity,
                showlegend=False,
                hoverinfo="skip",
            ), row=row, col=col)

    colors = [ELEMENT_COLORS.get(a.GetAtomicNum(), OTHER_COLOR) for a in mol.GetAtoms()]
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]

    if atom_weights is not None:
        opacities = [0.3 + 0.7 * w for w in atom_weights]
    else:
        opacities = [1.0] * mol.GetNumAtoms()

    # Draw each atom individually so it can have its own opacity
    for k in range(mol.GetNumAtoms()):
        fig.add_trace(go.Scatter(
            x=[coords[k, 0]], y=[coords[k, 1]],
            mode="markers+text",
            marker=dict(size=20, color=colors[k], opacity=opacities[k],
                        line=dict(width=1, color="white")),
            text=[symbols[k]],
            textposition="middle center",
            textfont=dict(size=8, color="white"),
            showlegend=False,
            hoverinfo="skip",
        ), row=row, col=col)


rng = np.random.default_rng(42)
sample_idx = rng.choice(num_mols, size=4, replace=False)

fig = make_subplots(
    rows=2, cols=2, horizontal_spacing=0.04, vertical_spacing=0.1,
    subplot_titles=[
        f"{mol_names[i]} ({'BBB+' if labels[i] else 'BBB-'})" for i in sample_idx
    ],
)
for k, mol_i in enumerate(sample_idx):
    r, c = divmod(k, 2)
    mol = Chem.MolFromSmiles(mol_smiles[mol_i])
    draw_molecule(fig, mol, r + 1, c + 1)

fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_layout(
    height=500, width=900, template="plotly_white",
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()

## Train/test split

Random 80/20 split over molecules. Since all molecules live in one disconnected graph, we just need boolean masks over molecule indices.

In [7]:
rng = np.random.default_rng(0)
perm = rng.permutation(num_mols)
split = int(0.8 * num_mols)

train_mask = jnp.zeros(num_mols, dtype=bool).at[perm[:split]].set(True)
test_mask = jnp.zeros(num_mols, dtype=bool).at[perm[split:]].set(True)

print(f"Train: {train_mask.sum()} molecules")
print(f"Test:  {test_mask.sum()} molecules")

Train: 1631 molecules
Test:  408 molecules


## Model

Two GATv2Conv layers followed by mean pooling and a linear classifier.

Each GATv2Conv layer passes messages along bonds. Every atom collects features from its neighbors, weighted by learned attention scores. Unlike a simple average over neighbors, attention lets the model decide which bonds matter most for each atom. The scores depend on both the atoms *and* the bond type between them (via edge features).

After two rounds of message passing, each atom's representation encodes its local chemical environment. Mean pooling aggregates all atom representations into a single vector per molecule, which the linear layer maps to a BBB prediction.

In [8]:
from jaxtyping import Array, Bool, Float, Int, PRNGKeyArray


class MoleculeClassifier(nn.Module):
    """Graph attention classifier for molecular property prediction."""

    gat_1: gnn.GATv2Conv
    gat_2: gnn.GATv2Conv
    classifier: nn.Linear

    def __init__(
        self,
        node_dim: int,
        hidden_dim: int,
        edge_dim: int,
        num_heads: int = 4,
        *,
        key: PRNGKeyArray,
    ) -> None:
        keys = jax.random.split(key, 3)
        self.gat_1 = gnn.GATv2Conv(
            node_dim, hidden_dim, num_heads=num_heads, edge_dim=edge_dim, key=keys[0],
        )
        self.gat_2 = gnn.GATv2Conv(
            hidden_dim, hidden_dim, num_heads=num_heads, edge_dim=edge_dim, key=keys[1],
        )
        self.classifier = nn.Linear(hidden_dim, 1, key=keys[2])

    def __call__(
        self,
        x: Float[Array, "n d"],
        senders: Int[Array, " e"],
        receivers: Int[Array, " e"],
        x_edge: Float[Array, "e f"],
        graph_index: Int[Array, " n"],
        num_graphs: int,
    ) -> Float[Array, " g"]:
        # Message passing
        x = jax.nn.elu(self.gat_1(x, senders, receivers, x_edge))
        x = self.gat_2(x, senders, receivers, x_edge)

        # Mean pool per molecule
        graph_sum = jax.ops.segment_sum(x, graph_index, num_graphs)
        counts = jax.ops.segment_sum(jnp.ones(x.shape[0]), graph_index, num_graphs)
        graph_emb = graph_sum / jnp.maximum(counts[:, None], 1.0)

        return self.classifier(graph_emb).squeeze(-1)


HIDDEN_DIM = 64
NUM_HEADS = 4

model = MoleculeClassifier(
    x.shape[1], HIDDEN_DIM, x_edge.shape[1], NUM_HEADS, key=jax.random.key(0),
)
print(f"Parameters: {model.num_params:,}")
model

Parameters: 10,177


MoleculeClassifier(
  gat_1=GATv2Conv(
    w_sender=Param(f32[9, 4, 16], trainable=True),
    w_receiver=Param(f32[9, 4, 16], trainable=True),
    att=Param(f32[4, 16], trainable=True),
    b=Param(f32[64], trainable=True),
    w_edge=Param(f32[4, 4, 16], trainable=True),
    num_heads=4,
    negative_slope=0.2,
    edge_dim=4,
  ),
  gat_2=GATv2Conv(
    w_sender=Param(f32[64, 4, 16], trainable=True),
    w_receiver=Param(f32[64, 4, 16], trainable=True),
    att=Param(f32[4, 16], trainable=True),
    b=Param(f32[64], trainable=True),
    w_edge=Param(f32[4, 4, 16], trainable=True),
    num_heads=4,
    negative_slope=0.2,
    edge_dim=4,
  ),
  classifier=Linear(
    w=Param(f32[64, 1], trainable=True),
    b=Param(f32[1], trainable=True),
  ),
)

## Training

Full-batch training with binary cross-entropy loss. Since all molecules live in a single disconnected graph, one forward pass processes the entire dataset.

In [9]:
def loss_fn(
    model: MoleculeClassifier,
    x: Float[Array, "n d"],
    senders: Int[Array, " e"],
    receivers: Int[Array, " e"],
    x_edge: Float[Array, "e f"],
    graph_index: Int[Array, " n"],
    labels: Float[Array, " g"],
    mask: Bool[Array, " g"],
) -> Float[Array, ""]:
    """Masked binary cross-entropy loss over labeled molecules."""
    logits = model(x, senders, receivers, x_edge, graph_index, num_mols)
    bce = optax.sigmoid_binary_cross_entropy(logits, labels)
    return jnp.where(mask, bce, 0.0).sum() / mask.sum()


@jax.jit
def train_step(model, optimizer, x, senders, receivers, x_edge, graph_index, labels, mask):
    """Compute gradients and apply optimizer update."""
    loss, grads = jax.value_and_grad(loss_fn)(
        model, x, senders, receivers, x_edge, graph_index, labels, mask,
    )
    model, optimizer = optimizer.update(model, grads)
    return model, optimizer, loss


@jax.jit
def evaluate(model, x, senders, receivers, x_edge, graph_index, labels, mask):
    """Compute loss and accuracy over masked molecules."""
    logits = model(x, senders, receivers, x_edge, graph_index, num_mols)
    bce = optax.sigmoid_binary_cross_entropy(logits, labels)
    loss = jnp.where(mask, bce, 0.0).sum() / mask.sum()
    preds = (jax.nn.sigmoid(logits) > 0.5).astype(jnp.float32)
    acc = jnp.where(mask, preds == labels, False).sum() / mask.sum()
    return loss, acc


LEARNING_RATE = 5e-3
NUM_EPOCHS = 100

model = MoleculeClassifier(
    x.shape[1], HIDDEN_DIM, x_edge.shape[1], NUM_HEADS, key=jax.random.key(0),
)
optimizer = ion.Optimizer(optax.adam(LEARNING_RATE), model)

train_losses, test_losses = [], []
train_accs, test_accs = [], []
eval_epochs = []

for epoch in tqdm(range(NUM_EPOCHS), desc="Training"):
    model, optimizer, _ = train_step(
        model, optimizer, x, senders, receivers, x_edge, graph_index, labels, train_mask,
    )

    if epoch % 10 == 0 or epoch == NUM_EPOCHS - 1:
        train_loss, train_acc = evaluate(
            model, x, senders, receivers, x_edge, graph_index, labels, train_mask,
        )
        test_loss, test_acc = evaluate(
            model, x, senders, receivers, x_edge, graph_index, labels, test_mask,
        )
        train_losses.append(train_loss.item())
        test_losses.append(test_loss.item())
        train_accs.append(train_acc.item())
        test_accs.append(test_acc.item())
        eval_epochs.append(epoch)

print(f"Train accuracy: {train_accs[-1]:.2%}")
print(f"Test accuracy:  {test_accs[-1]:.2%}")

Training: 100%|██████████| 100/100 [01:05<00:00,  1.52it/s]

Train accuracy: 81.79%
Test accuracy:  78.43%


In [10]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss", "Accuracy"])

fig.add_trace(go.Scatter(
    x=eval_epochs, y=train_losses, mode="lines", name="Train",
    line=dict(width=5, color="#636EFA"), opacity=0.8,
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=eval_epochs, y=test_losses, mode="lines", name="Test",
    line=dict(width=5, color="#EF553B"), opacity=0.8,
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=eval_epochs, y=train_accs, mode="lines", name="Train",
    line=dict(width=5, color="#636EFA"), opacity=0.8, showlegend=False,
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=eval_epochs, y=test_accs, mode="lines", name="Test",
    line=dict(width=5, color="#EF553B"), opacity=0.8, showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="BCE loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2)
fig.update_layout(height=400, template="plotly_white")
fig.show()

## ROC curve

In [11]:
logits_all = model(x, senders, receivers, x_edge, graph_index, num_mols)
probs_all = np.array(jax.nn.sigmoid(logits_all))

test_mask_np = np.array(test_mask)
test_probs = probs_all[test_mask_np]
test_labels = np.array(labels)[test_mask_np]

# Sweep thresholds to compute ROC
thresholds = np.linspace(0, 1, 200)
tpr, fpr = [], []
for t in thresholds:
    pred = test_probs >= t
    tp = (pred & (test_labels == 1)).sum()
    fp = (pred & (test_labels == 0)).sum()
    fn = (~pred & (test_labels == 1)).sum()
    tn = (~pred & (test_labels == 0)).sum()
    tpr.append(tp / max(tp + fn, 1))
    fpr.append(fp / max(fp + tn, 1))

tpr, fpr = np.array(tpr), np.array(fpr)

# AUC via trapezoidal rule
order = np.argsort(fpr)
fpr_sorted, tpr_sorted = fpr[order], tpr[order]
auc = np.sum((fpr_sorted[1:] - fpr_sorted[:-1]) * (tpr_sorted[1:] + tpr_sorted[:-1]) / 2)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    name=f"GATv2 (AUC = {auc:.3f})",
    line=dict(width=5, color="#636EFA"), opacity=0.8,
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random",
    line=dict(width=2, color="rgb(96,96,96)", dash="dash"),
))
fig.update_layout(
    title="ROC curve (test set)",
    xaxis_title="False positive rate",
    yaxis_title="True positive rate",
    height=450, width=500, template="plotly_white",
)
fig.show()

## Attention weights

We can look inside the trained model to see which bonds it pays attention to. To extract the weights, we replay the second GATv2Conv layer's computation step by step, matching `GATv2Conv.__call__` exactly:

1. Project node features with separate sender and receiver weight matrices
2. Combine projections at each edge, add projected edge features
3. Apply LeakyReLU, then dot with the attention vector to get a score per edge
4. Normalize scores per receiving node via softmax (so each node's incoming attention sums to 1)

In [12]:
def extract_attention(layer, x_in, senders, receivers, x_edge=None):
    """Replay GATv2Conv forward pass to extract attention weights."""
    n = x_in.shape[0]

    x_s = jnp.einsum("ni, ihk -> nhk", x_in, layer.w_sender)
    x_r = jnp.einsum("ni, ihk -> nhk", x_in, layer.w_receiver)

    edge_h = x_s[senders] + x_r[receivers]

    if x_edge is not None and layer.w_edge is not None:
        edge_proj = jnp.einsum("ef, fhk -> ehk", x_edge, layer.w_edge)
        edge_h = edge_h + edge_proj

    logits = jnp.einsum(
        "ehk, hk -> eh",
        jax.nn.leaky_relu(edge_h, layer.negative_slope),
        layer.att,
    )

    return gnn.segment_softmax(logits, receivers, n)


# Intermediate features after layer 1
x_hidden = jax.nn.elu(model.gat_1(x, senders, receivers, x_edge))

# Attention from layer 2
attn = extract_attention(model.gat_2, x_hidden, senders, receivers, x_edge)
print(f"Attention shape: {attn.shape}  (edges, heads)")

Attention shape: (154910, 4)  (edges, heads)


## Attention on molecules

Attention weights overlaid on molecular structure. Bond thickness and opacity encode per-bond attention, while atom opacity encodes the mean attention of each atom's incident bonds. The subtitle shows the model's predicted probability and the true label (BBB+ = penetrates the barrier, BBB- = does not).

In [13]:
def molecule_attention(mol_idx, mol, attn, senders, receivers, graph_index, head=None):
    """Get per-bond and per-atom attention weights for a molecule, normalized to [0, 1]."""
    node_mask = graph_index == mol_idx
    node_start = int(jnp.argmax(node_mask))

    attn_vals = np.array(attn[:, head] if head is not None else attn.mean(axis=1))
    senders_np = np.array(senders)
    receivers_np = np.array(receivers)

    bond_attn = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx() + node_start
        j = bond.GetEndAtomIdx() + node_start
        fwd = attn_vals[(senders_np == i) & (receivers_np == j)]
        bwd = attn_vals[(senders_np == j) & (receivers_np == i)]
        bond_attn.append(np.concatenate([fwd, bwd]).mean())

    bond_attn = np.array(bond_attn)
    if bond_attn.max() > bond_attn.min():
        bond_attn = (bond_attn - bond_attn.min()) / (bond_attn.max() - bond_attn.min())

    # Per-atom: mean attention of incident bonds
    atom_attn = np.zeros(mol.GetNumAtoms())
    counts = np.zeros(mol.GetNumAtoms())
    for idx, bond in enumerate(mol.GetBonds()):
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        atom_attn[i] += bond_attn[idx]
        atom_attn[j] += bond_attn[idx]
        counts[i] += 1
        counts[j] += 1
    atom_attn /= np.maximum(counts, 1)
    if atom_attn.max() > atom_attn.min():
        atom_attn = (atom_attn - atom_attn.min()) / (atom_attn.max() - atom_attn.min())

    return bond_attn, atom_attn


# Pick 4 test molecules
rng = np.random.default_rng(7)
test_indices = np.where(np.array(test_mask))[0]
sample_test = rng.choice(test_indices, size=4, replace=False)

fig = make_subplots(
    rows=2, cols=2, horizontal_spacing=0.04, vertical_spacing=0.1,
    subplot_titles=[
        f"{mol_names[i]} - p={probs_all[i]:.2f} ({'BBB+' if labels[i] else 'BBB-'})"
        for i in sample_test
    ],
)

for k, mol_idx in enumerate(sample_test):
    r, c = divmod(k, 2)
    mol = Chem.MolFromSmiles(mol_smiles[mol_idx])
    bond_weights, atom_weights = molecule_attention(
        mol_idx, mol, attn, senders, receivers, graph_index,
    )
    draw_molecule(fig, mol, r + 1, c + 1, bond_weights=bond_weights,
                  atom_weights=atom_weights, show_bond_types=False)

fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_layout(
    height=500, width=900, template="plotly_white",
    margin=dict(l=10, r=10, t=40, b=10),
)
fig.show()

## Multi-head attention

GATv2Conv uses multiple attention heads (4 in our model), each with independent learned weights. Different heads can specialize on different chemical patterns. Here we show the same molecule through each head separately.

In [14]:
mol_idx = sample_test[1]
mol = Chem.MolFromSmiles(mol_smiles[mol_idx])

fig = make_subplots(
    rows=2, cols=2, horizontal_spacing=0.04, vertical_spacing=0.1,
    subplot_titles=[f"Head {h + 1}" for h in range(NUM_HEADS)],
)

for h in range(NUM_HEADS):
    r, c = divmod(h, 2)
    bond_weights, atom_weights = molecule_attention(
        mol_idx, mol, attn, senders, receivers, graph_index, head=h,
    )
    draw_molecule(fig, mol, r + 1, c + 1, bond_weights=bond_weights,
                  atom_weights=atom_weights, show_bond_types=False)

fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_layout(
    title=f"{mol_names[mol_idx]} - per-head attention",
    height=500, width=900, template="plotly_white",
    margin=dict(l=10, r=10, t=60, b=10),
)
fig.show()

## Learned atom embeddings

After two rounds of message passing, each atom has a 64-dimensional embedding that captures its chemical context, not just its element type, but what it's bonded to. We project these down to 2D with PCA to visualize the learned chemical space.

In [15]:
# Embeddings after both GATv2Conv layers
x_emb = np.array(model.gat_2(x_hidden, senders, receivers, x_edge))

# PCA via SVD (no sklearn needed)
x_centered = x_emb - x_emb.mean(axis=0)
u, s, vt = np.linalg.svd(x_centered, full_matrices=False)
pca = u[:, :2] * s[:2]

# Subsample for plotting
rng = np.random.default_rng(0)
max_points = 5000
idx = rng.choice(pca.shape[0], size=min(max_points, pca.shape[0]), replace=False)
pca_sub = pca[idx]
elem_idx = np.array(x)[idx].argmax(axis=1)

fig = go.Figure()
for i, name in enumerate(ELEMENT_NAMES):
    mask = elem_idx == i
    if mask.sum() == 0:
        continue
    fig.add_trace(go.Scatter(
        x=pca_sub[mask, 0], y=pca_sub[mask, 1],
        mode="markers", name=name,
        marker=dict(size=5, color=ELEMENT_PALETTE[i], opacity=0.5),
    ))

fig.update_layout(
    title="Atom embeddings (PCA)",
    xaxis_title="PC 1", yaxis_title="PC 2",
    height=500, width=600, template="plotly_white",
)
fig.show()